## Simple test of `POPSRegression`

Comparing uncertainty estimates from `BayesianRidge` (epistemic only) vs `POPSRegression` (epistemic + misspecification) for a misspecified polynomial surrogate model fit to low-noise data.

Usage is very similar to `sklearn.linear_model.BayesianRidge`.

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from popsregression import POPSRegression

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def target_function(x):
    return (x**3 + 0.01 * x**4) * 0.1 + np.sin(x) * x * 10.0


def generate_data(N):
    x_train = np.sort(
        np.append(np.random.uniform(-1, 1, N), np.linspace(-1, 1, 2)) * 10
    )
    x_dense = np.linspace(-1.1, 1.1, 51) * 10
    y_dense = target_function(x_dense)

    p = PolynomialFeatures(degree=4, include_bias=True)
    X_train = p.fit_transform(x_train.reshape(-1, 1))
    X_dense = p.fit_transform(x_dense.reshape(-1, 1))
    y_train = target_function(x_train)
    return X_train, x_train, y_train, X_dense, x_dense, y_dense


def plot_panel(ax, x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=None, y_min=None):
    if y_max is not None and y_min is not None:
        ax.fill_between(x_dense, y_min, y_max, alpha=0.2, facecolor="0.5",
                        label="max/min")
    else:
        ax.fill_between(x_dense, y_pred - 4 * y_std, y_pred + 4 * y_std,
                        alpha=0.2, facecolor="0.5", label=r"$\pm4\sigma$")
    ax.fill_between(x_dense, y_pred - 2 * y_std, y_pred + 2 * y_std,
                    alpha=0.5, facecolor="C1", label=r"$\pm2\sigma$")
    ax.plot(x_dense, y_pred, "C1-", lw=4)
    ax.plot(x_train, y_train, "b.", label="Train")
    ax.plot(x_dense, y_dense, "k-")

### Compare BayesianRidge, POPS Ensemble, and POPS Hypercube

Fitting a quartic polynomial (P=5) to a complex oscillatory function with increasing N/P ratio. BayesianRidge epistemic uncertainty vanishes with more data, while POPS maintains honest uncertainty where the polynomial deviates from the truth.

In [ ]:
np.random.seed(42)

titles = ["Bayesian Ridge", "POPS Ensemble", "POPS Hypercube"]
fig, axs = plt.subplots(3, 3, figsize=(8, 7), sharex=True, sharey=True)
N_array = [10, 50, 500]
P = 5

for i, N in enumerate(N_array):
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)

    ens = POPSRegression(leverage_percentile=0.0, posterior="ensemble")
    hyc = POPSRegression(leverage_percentile=0.0, posterior="hypercube")
    bay = BayesianRidge(fit_intercept=False)

    ens.fit(X_train, y_train)
    hyc.fit(X_train, y_train)
    bay.fit(X_train, y_train)

    # BayesianRidge - epistemic only (no aleatoric alpha_)
    b_pred = bay.predict(X_dense, return_std=False)
    b_std = np.sqrt(np.sum(np.dot(X_dense, bay.sigma_) * X_dense, axis=1))
    plot_panel(axs[0, i], x_dense, y_dense, x_train, y_train, b_pred, b_std)

    # POPS Ensemble
    y_pred, y_std, y_max, y_min = ens.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[1, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    # POPS Hypercube
    y_pred, y_std, y_max, y_min = hyc.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[2, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    axs[0, i].set_title(f"N/P = {N // P}")

for j, title in enumerate(titles):
    axs[j, 0].set_ylim(-250, 250)
    axs[j, 0].set_xlim(-10, 10)
    axs[j, 1].legend(fontsize=9, loc="lower center")
    axs[j, 0].set_ylabel(title, fontsize=13)

plt.tight_layout()
plt.savefig("example_image.png")
plt.show()

### Observations

- **N/P = 2**: Both methods show wide uncertainty, but POPS provides better coverage of the true function (black line).
- **N/P = 10**: BayesianRidge epistemic uncertainty has already shrunk significantly. POPS correctly maintains wider bands where the polynomial deviates from the truth.
- **N/P = 100**: BayesianRidge epistemic uncertainty is nearly invisible, yet the polynomial still cannot match the oscillatory target. POPS maintains honest uncertainty reflecting this structural limitation.

The **hypercube** posterior (bottom row) tends to give the most conservative bounds, while the **ensemble** posterior (middle row) directly uses the pointwise corrections.